# 📊 Baseline Models

This notebook loads the cleaned data from *01b* and trains two simple baseline regressors: **Linear Regression** and **Random Forest** (default hyper‑parameters). It reports MAE, RMSE and R² for each model on the same test set.

In [ ]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.svm import SVR
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROCESSED_DIR = Path("../data/processed")
df = pd.read_csv(PROCESSED_DIR / 'car-price-clean.csv')
X = df.drop('selling_price', axis=1)
y = df['selling_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)


In [ ]:
# Dictionnaire regroupant les modèles à tester
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_test_lr = scaler.transform(X_test)
models = {
    "Régression Linéaire": LinearRegression(),
    # "Ridge": Ridge(),
    # "Arbre de Décision": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    # "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(random_state=42, n_estimators=100),
    "Svr" : SVR(kernel='rbf', C=1.0, epsilon=0.1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({"Modèle": name, "MAE": mae, "RMSE": rmse, "R2 Score": r2})

# Conversion en DataFrame pour un affichage lisible
df_results = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
df_results